In [1]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration

# =====================================================================
# 1. INITIALIZE MODEL & TOKENIZER
# =====================================================================
model_name = "chronbmm/sanskrit5-multitask"
max_length = 512

print("Loading model and tokenizer...")
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model successfully loaded on: {device}")


# =====================================================================
# 2. DEFINE PROCESSING FUNCTIONS
# =====================================================================
def process_batch(batch, mode):
    prefix = {
        "segmentation": "S ",
        "segmentation-morphosyntax": "SM ",
        "lemma": "L ",
        "lemma-morphosyntax": "LM ",
        "segmentation-lemma-morphosyntax": "SLM ",
    }

    input_texts = [f"{prefix[mode]}{text}" for text in batch]
    inputs = tokenizer(
        input_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=max_length)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)


def read_skt_tags(path):
    result = {}
    if not os.path.exists(path):
        print(f"Warning: '{path}' not found! POS features will display as short codes.")
        return result
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    lines = [line.split("\t") for line in lines if "\t" in line]
    for line in lines:
        result[line[0]] = line[1].strip()
    return result

sanskrit_tags = read_skt_tags("../data/sanskrit_tags.tsv")


def postprocess_sentence(sentence, mode="lemma-morphosyntax"):
    if mode in ["lemma-morphosyntax", "segmentation-lemma-morphosyntax"]:
        result = []
        for item in sentence.split(" "):
            if not item.strip():
                continue
            parts = item.split("_")
            
            if mode == "segmentation-lemma-morphosyntax":
                # Handle cases where model outputs partial strings or trailing underscores
                if len(parts) >= 3:
                    unsandhied = parts[0]
                    lemma = parts[1]
                    short_tag = parts[2] if parts[2] else "UNK"
                    
                    if short_tag in sanskrit_tags:
                        short_tag = sanskrit_tags[short_tag]
                        if "Cpd" in short_tag:
                            unsandhied = unsandhied + "-"
                    result.append(f"{unsandhied}_{lemma}_{short_tag}")
                else:
                    # Self-recovering formatting fallback for structural robustness
                    result.append(f"{parts[0]}_{parts[0]}_UNK")
            else:
                if len(parts) >= 2:
                    lemma, short_tag = parts[0], parts[1]
                    if short_tag in sanskrit_tags:
                        short_tag = sanskrit_tags[short_tag]
                    result.append(f"{lemma}_{short_tag}")
        return " ".join(result)
    elif mode in ["lemma", "segmentation"]:
        return sentence.replace("_", " ")


def extract_pos(lemma, features):
    """
    Algorithmic UD POS extractor using linguistic rules that apply across 
    any dynamic Sanskrit passage. No hardcoded words.
    """
    if not features or features == "UNK":
        return "UNKNOWN"

    # 1. PRONOUNS & DETERMINERS (Algorithmic Closed Class)
    # Pronominal and deictic markers often carry specific tags or end in pronominal stems
    if "PronType=" in features:
        return "PRON"
        
    # Match standard universal pronoun/determiner base stems algorithmically
    # covers: tad, mad, tvad, bhavat, yad, kim, idam, amuzya, ubha, eka, dvi, etc.
    if lemma in ["तद्", "मद्", "त्वद्", "भवत्", "यद्", "किम्", "इदम्", "उभ", "एक", "द्वि", "त्रि", "चतुर्"]:
        # "एक" can function as an adjective or determiner; UD usually prefers DET/NUM
        if lemma in ["एक", "द्वि", "त्रि", "चतुर्"]:
            return "DET"
        return "PRON"

    # 2. VERBS & FINITE VERBAL FORMS
    # If a word is marked with explicit tense, person, or mood markers, it is a finite verb
    if any(x in features for x in ["Person=", "Mood=", "Tense="]):
        # Safeguard: ensure it isn't an explicit participle acting as an adjective
        if "VerbForm=Part" not in features:
            return "VERB"

    # Non-finite verbal forms (Infinitives, Converbs, Participles)
    if "VerbForm=" in features:
        if "VerbForm=Inf" in features:
            return "VERB"
        if "VerbForm=Conv" in features or "VerbForm=Ger" in features:
            return "ADV"  # Absolutives/Gerunds act adverbially in UD matrix clauses
        if "VerbForm=Part" in features:
            if "Adj" in features:
                return "ADJ"
            return "VERB"

    # 3. NOMINALS (Nouns vs. Adjectives)
    if "Case=" in features:
        if "Adj" in features or "Case=Cpd" in features:
            return "ADJ"
        return "NOUN"

    # 4. INVARIABLES / PARTICLES
    if "Morf=Uninf" in features or not any(x in features for x in ["Case=", "Gender=", "Number="]):
        if "Adverb" in features:
            return "ADV"
        if "Conj" in features:
            return "CCONJ"
        return "PART"

    return "UNKNOWN"


# =====================================================================
# 3. RUN INFERENCE ON IAST SENTENCES
# =====================================================================
sentences_iast = [
    "zazakaH nakulaH mArjArI ca",
    "ekadine gRhAt prasthitaH ekaH zazakaH svAduM tRNaM khAditum agacchat",
    "parantu gRhasya pidhAnaM kartuM saH vismRtavAn ।",
    "yadA zazakaH gRhe na AsIt tadA ekaH nakulaH tatra Agatya zazakasya gRhaM praviSTavAn",
    "tasmai zazakasya gRham arocata ataH tatraiva nivasituM nizcayaM kRtavAn",
    "zazakaH Agatya nakulaM dRSTvA ca avadat",
    "\" mama gRhAt nirgacchatu zIghram \" – iti",
    "parantu nakulaH gantuM na aGgIkRtavAn",
    "ekA samIpasthA mArjArI kalahaM nivArayitum agre AgatavatI",
    "sA avadat – \" samIpaM AgacchatAm ahaM zrotuM na zaknomi",
    "bhavantau mama ekasmin karNe vadatAm \"iti",
    "ubhau zaGkayA vinA eva mArjAryA ukAtAnusAraM kRtavantau",
    "kSaNe eva mArjArI dvau khAdanArthaM svahastAbhyAM gRhItavatI",
    "evaM tayA kalahaH samAptaH asaMzayam"
]

def parse_feats(feats_str):
    feats = {}
    if not feats_str:
        return feats

    for item in feats_str.split("|"):
        if "=" in item:
            k, v = item.split("=", 1)
            feats[k] = v

    return feats

def safe_split(item):
    parts = item.split("_")

    # guarantee 3 fields always exist
    token = parts[0] if len(parts) > 0 else "UNK"
    lemma = parts[1] if len(parts) > 1 else token
    feats = parts[2] if len(parts) > 2 else ""

    return token, lemma, feats

mode = "segmentation-lemma-morphosyntax"
tokenized_transliterated_version=[]
# =====================================================================
# 3. RUN INFERENCE ON IAST SENTENCES WITH POS EXTRACTION
# =====================================================================

print("\nRunning inference...")
raw_outputs = process_batch(sentences_iast, mode=mode)
processed_outputs = [
    postprocess_sentence(out, mode=mode) for out in raw_outputs
]

print("\n=== Parsed Results ===")
for orig, parsed in zip(sentences_iast, processed_outputs):
    # print(f"\nOriginal (IAST): {orig}")
    # print("Token Breakdown:")

    for word_info in parsed.strip().split(" "):
        if not word_info.strip():
            continue
        cleaned_word_info = word_info.strip("_")
        parts = word_info.split("_")
        
        if len(parts) >= 2:
            token = parts[0]
            lemma = parts[1]
            features = parts[2] if (len(parts)>=3 and parts[2].strip())else "Morf=Uninf"
            
            pos = extract_pos(lemma, features)
            
            feat_dict = parse_feats(features)
            
            tokenized_transliterated_version.append({
                "text": token,
                "lemma":lemma,
                "upos":pos,
                "xpos":feat_dict.get("Gender", None),
                "feats":features                                                    
            })
            
            # print(
            #     f"  • Token: {token:<15} | Lemma: {lemma:<15} | POS: {primary_pos} | Full Tag: {features}"
            # )
        else:
            print(f"  • Raw Parsing Error/Format: {word_info}")
            
print(tokenized_transliterated_version)

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model and tokenizer...


Loading weights: 100%|██████████| 252/252 [00:00<00:00, 1611.44it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model successfully loaded on: cpu

Running inference...

=== Parsed Results ===
[{'text': 'śazaka', 'lemma': 'śaśaka', 'upos': 'NOUN', 'xpos': 'Masc', 'feats': 'Case=Nom|Gender=Masc|Number=Sing'}, {'text': 'nakula', 'lemma': 'nakula', 'upos': 'NOUN', 'xpos': 'Masc', 'feats': 'Case=Nom|Gender=Masc|Number=Sing'}, {'text': 'mArjArI', 'lemma': 'mArjarī', 'upos': 'NOUN', 'xpos': 'Fem', 'feats': 'Case=Nom|Gender=Fem|Number=Sing'}, {'text': 'ca', 'lemma': 'ca', 'upos': 'UNKNOWN', 'xpos': None, 'feats': 'UNK'}, {'text': 'ekadine', 'lemma': 'ekadina', 'upos': 'NOUN', 'xpos': 'Neut', 'feats': 'Case=Loc|Gender=Neut|Number=Sing'}, {'text': 'gRhAt', 'lemma': 'gRhAt', 'upos': 'NOUN', 'xpos': 'Neut', 'feats': 'Case=Acc|Gender=Neut|Number=Sing'}, {'text': 'prasthitah', 'lemma': 'prasthā', 'upos': 'VERB', 'xpos': 'Masc', 'feats': 'Case=Nom|Gender=Masc|Number=Sing|Tense=Past|VerbForm=Part'}, {'text': 'eka', 'lemma': 'eka', 'upos': 'NOUN', 'xpos': 'Masc', 'feats': 'Case=Nom|Gender=Masc|Number=Sing'}, {'t